In [1]:
# Import packages
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import plotly.express as px

# Set options
pd.set_option('display.max_columns', 55)

In [2]:
# Read pre_processed csv
clean_data = pd.read_parquet("sample_by_gene.parquet")

In [3]:
# Standardize and run PCA
X = clean_data.drop(columns=['Target(SNHG14)', 'sample_type']).values

pca = PCA(n_components=2)
pcs = pca.fit_transform(StandardScaler().fit_transform(X))


In [4]:
# Build plot dataframe
pca_df = pd.DataFrame(pcs, columns=['PC1', 'PC2'], index=clean_data.index)
pca_df['sample_type'] = clean_data['sample_type'].values

fig = px.scatter(
    pca_df,
    x='PC1', y='PC2',
    color='sample_type',
    hover_name=pca_df.index,
    labels={
        'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
        'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    },
    title='PCA of Gene Expression by Sample Type',
)

# Control traces: large star with black border; all others: small circle
fig.for_each_trace(
    lambda t: t.update(marker_size=14, marker_symbol='star', marker_line_width=2, marker_line_color='black')
    if t.name == 'Control' else t.update(marker_size=7)
)

fig.show()

In [5]:
fig2 = px.scatter(
    pca_df,
    x='PC1', y='PC2',
    color='sample_type',
    text='sample_type',
    hover_name=pca_df.index,
    labels={
        'PC1': f'PC1 ({pca.explained_variance_ratio_[0]:.1%})',
        'PC2': f'PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    },
    title='PCA of Gene Expression by Sample Type',
)

fig2.update_traces(mode='text')
fig2.update_layout(showlegend=False)

for trace in fig2.data:
    if trace.name == 'Control':
        trace.textfont.color = 'black'
        trace.textfont.size = 14
        trace.textfont.weight = 'bold'
    else:
        trace.textfont.color = trace.marker.color
        trace.textfont.size = 10

fig2.show()